# Generating synthetic weather
**Introduction to Climate Modeling · November 12, 2026**

Can we learn a probability model that generates new weather with a realistic climate?

Use the NYC temperature data from Lab 0c. Fit a seasonal cycle and weather variability to earlier years, then generate new daily sequences in two ways: independent samples and samples with day-to-day persistence. Compare their statistics with later observations.

**Allow about 25–35 minutes.** All code is supplied. Run the cells in order, make the requested guesses, and write brief answers at the four prompts. The optional final experiment can be saved for later.

Download the [notebook and data](https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/generative_weather.zip), unzip the folder, and open `generative_weather.ipynb` in the course [LEAP Jupyter environment](https://leap.2i2c.cloud/). Keep the `data` folder beside it. The package runs offline once the Python libraries are available.

This is a simple **statistical generative model**. We learn its seasonal distribution and persistence from observations. The goal is plausible weather statistics across many generated sequences. Each sequence is a new realization, so its individual hot and cold days will differ from the observed sequence.

In [ ]:
from pathlib import Path
from urllib.request import urlopen
import hashlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.figsize": (9, 4), "font.size": 11,
                     "axes.spines.top": False, "axes.spines.right": False})

## 1. Reuse the NYC observations

The observations are daily Central Park temperature from NOAA, calculated as the average of the day's maximum and minimum temperature, in °C. We use only `t_obs` from the original Lab 0c file.

- **Fit the models:** 1995–2009.
- **Evaluate their generated climate:** 2010–2014.

The seasonal cycle, variability, persistence and hot-day threshold will all be learned from the training years. We will not use the later observations to fit them. Source details are in `data/provenance.json`.

In [ ]:
data_file = Path("data/centralpark_obs_era5_1995-2014.csv")
SOURCE_URL = 'https://raw.githubusercontent.com/dhruvbalwada/intro-climate-modeling-fall2026/a44a0f875c055ec6cde6d0c48b970687428040df/labs/data/centralpark_obs_era5_1995-2014.csv'
SOURCE_SHA256 = 'dc320f02b00dd82dcb7150683fe29fcfe5fe7d7d22477b85d6965819588a2d4e'

if not data_file.exists():
    print("Downloading the fixed classroom data file...")
    with urlopen(SOURCE_URL, timeout=45) as response:
        raw = response.read()
    if hashlib.sha256(raw).hexdigest() != SOURCE_SHA256:
        raise ValueError("Data do not match the class version. Use the supplied data folder.")
    data_file.parent.mkdir(parents=True, exist_ok=True)
    data_file.write_bytes(raw)

frame = pd.read_csv(data_file, index_col=0, parse_dates=True)
observed = frame["t_obs"].sort_index()
expected_dates = pd.date_range("1995-01-01", "2014-12-31", freq="D")
if not observed.index.equals(expected_dates) or observed.isna().any():
    raise ValueError("This exercise expects the supplied complete daily observation record.")

train = observed.loc[:"2009-12-31"]
test = observed.loc["2010-01-01":]
dates = test.index
print(f"Training: {len(train)} days; held out: {len(test)} days")

fig, ax = plt.subplots(figsize=(10, 3.8), layout="constrained")
ax.plot(train.index, train, color="#777777", lw=0.6, label="Training observations")
ax.plot(test.index, test, color="#111111", lw=0.6, label="Held-out observations")
ax.axvline(dates[0], color="#c54b26", ls="--", lw=1)
ax.set(ylabel="Daily temperature (°C)", title="The NYC temperature record from Lab 0c")
ax.legend(loc="upper left")
plt.show()

## 2. Learn the seasonal distribution

As in Lab 0c, represent the mean seasonal cycle with a smooth annual sine and cosine:

$$\mu(d)=b_0+b_1\sin(2\pi q(d))+b_2\cos(2\pi q(d)).$$

Here $q(d)$ is the fraction of the calendar year elapsed at the middle of day $d$. A sine and cosine together allow the amplitude and phase to be fitted. The model has three fitted coefficients.

Subtract this cycle from the training observations. The remaining variations are **weather anomalies**. Estimate their standard deviation in each month, then interpolate smoothly between months to obtain $\sigma(d)$.

The code below supplies these calculations. Inspect the mean cycle and the spread around it.

In [ ]:
def year_fraction(index):
    return (index.dayofyear.to_numpy() - 0.5) / (365 + index.is_leap_year.astype(int))

def seasonal_features(index):
    phase = 2*np.pi*year_fraction(index)
    return np.column_stack((np.ones(len(index)), np.sin(phase), np.cos(phase)))

# All fitted quantities below use training observations only.
mean_coefficients = np.linalg.lstsq(seasonal_features(train.index), train.to_numpy(), rcond=None)[0]

def mean_cycle(index):
    return seasonal_features(index) @ mean_coefficients

train_residual = train - mean_cycle(train.index)
monthly_sd = train_residual.groupby(train_residual.index.month).std().reindex(range(1, 13)).to_numpy()

def weather_sd(index):
    month_centres = (np.arange(12) + 0.5) / 12
    return np.interp(year_fraction(index), month_centres, monthly_sd, period=1.0)

mu = mean_cycle(dates)
sigma = weather_sd(dates)
obs_z = (test.to_numpy() - mu) / sigma

calendar = pd.date_range("2011-01-01", "2011-12-31")
phase_days = 365*year_fraction(train.index)
fig, ax = plt.subplots(figsize=(9, 4), layout="constrained")
ax.scatter(phase_days, train, s=3, color="#888888", alpha=0.12, label="Training observations")
mid = mean_cycle(calendar)
spread = weather_sd(calendar)
ax.plot(np.arange(365)+0.5, mid, color="#111111", lw=2, label="Learned seasonal mean")
ax.fill_between(np.arange(365)+0.5, mid-spread, mid+spread, color="#2374a5", alpha=0.3,
                label="Mean ± one modeled weather standard deviation")
ax.set(xlabel="Position in the year (days)", ylabel="Daily temperature (°C)", xlim=(0, 365))
ax.legend(fontsize=9, loc="upper left")
plt.show()

<a id="response-mean"></a>

**Discuss:** What would be missing if we generated a year using only the learned seasonal mean?

**Answer:** [Your answer here.]

## 3. Generate independent daily weather

For every day, draw a standard normal random number $z_d$. Convert it to temperature using the learned mean and variability:

$$z_d\sim N(0,1),\qquad T_d=\mu(d)+\sigma(d)z_d.$$

Different random draws create different weather sequences without refitting the model. We generate **200 members**, each covering the five held-out calendar years. This gives many realizations with which to estimate the generator's statistics.

The function below also has a persistence setting `phi`. Start with `phi=0`, which makes the days independent. We will interpret the nonzero version next.

In [ ]:
N_MEMBERS = 200
SEED = 12

def generate_weather(phi, n_members=N_MEMBERS, seed=SEED):
    if not -1 < phi < 1:
        raise ValueError("Use -1 < phi < 1 for this stationary generator.")
    rng = np.random.default_rng(seed)
    noise = rng.standard_normal((n_members, len(dates)))
    z = np.empty_like(noise)
    z[:, 0] = noise[:, 0]  # A draw from the stationary N(0, 1) distribution.
    for day in range(1, len(dates)):
        z[:, day] = phi*z[:, day-1] + np.sqrt(1-phi**2)*noise[:, day]
    temperature = mu[None, :] + sigma[None, :]*z
    return temperature, z

iid_temperature, iid_z = generate_weather(phi=0)

summer = (dates >= "2011-06-01") & (dates <= "2011-08-31")
fig, ax = plt.subplots(figsize=(10, 3.8), layout="constrained")
ax.plot(dates[summer], test.to_numpy()[summer], color="#111111", lw=1.3, label="Observed weather")
ax.plot(dates[summer], iid_temperature[0, summer], color="#2374a5", lw=1.1,
        label="One independent generated sequence")
ax.plot(dates[summer], mu[summer], color="#777777", ls="--", label="Learned seasonal mean")
ax.set(ylabel="Daily temperature (°C)", title="A new realization of summer weather")
ax.legend(fontsize=9)
plt.show()

<a id="response-persistence"></a>

**Guess before running:** Will drawing every day independently reproduce the lengths of observed hot and cold episodes? What feature of the time series would you inspect?

**Answer:** [Your answer here.]

## 4. Learn day-to-day persistence

Normalize the training anomalies by the learned seasonal standard deviation. Fit the coefficient $\phi$ relating one day's standardized anomaly to the previous day's:

$$\widehat\phi=\frac{\sum_d z_{d-1}z_d}{\sum_d z_{d-1}^2}.$$

Generate new anomalies with a first-order autoregressive model:

$$z_d=\phi z_{d-1}+\sqrt{1-\phi^2}\,\epsilon_d,\qquad \epsilon_d\sim N(0,1).$$

The factor $\sqrt{1-\phi^2}$ preserves unit variance in a stationary sequence. Thus changing $\phi$ changes the persistence while keeping the same normal marginal distribution. We still use the same $\mu(d)$ and $\sigma(d)$ to obtain temperatures.

In [ ]:
z_train = train_residual.to_numpy() / weather_sd(train.index)
phi_fit = np.dot(z_train[:-1], z_train[1:]) / np.dot(z_train[:-1], z_train[:-1])
print(f"Learned persistence coefficient phi = {phi_fit:.3f}")

persistent_temperature, persistent_z = generate_weather(phi=phi_fit)

fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True, sharey=True, layout="constrained")
for ax, values, title, color in zip(
        axes,
        [test.to_numpy(), iid_temperature[0], persistent_temperature[0]],
        ["Observed weather", "Independent generator", "Generator with learned persistence"],
        ["#111111", "#2374a5", "#c54b26"]):
    ax.plot(dates[summer], values[summer], color=color, lw=1.3)
    ax.plot(dates[summer], mu[summer], color="#777777", ls="--", lw=1)
    ax.set(title=title, ylabel="°C")
axes[-1].set_xlabel("Summer 2011 calendar dates")
plt.show()

## 5. Evaluate distributions and time dependence

First compare the distributions of standardized weather anomalies. Then compare correlations between anomalies separated by 1–10 days. These are correlations of **weather anomalies after removing the learned seasonal cycle**, so the annual cycle does not dominate the result.

The histograms pool the generated members; the correlation curves average the correlations calculated for each member. The observation curves use the five held-out years. This is a check of statistical properties, rather than a point-by-point forecast score.

In [ ]:
def lag_correlation(values, lag=1):
    values = np.atleast_2d(values)
    a, b = values[:, :-lag], values[:, lag:]
    a = a - a.mean(axis=1, keepdims=True)
    b = b - b.mean(axis=1, keepdims=True)
    return (a*b).sum(axis=1) / np.sqrt((a*a).sum(axis=1)*(b*b).sum(axis=1))

lags = np.arange(1, 11)
fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
bins = np.linspace(-4, 4, 41)
for values, label, color in [(obs_z, "Held-out observations", "#111111"),
                              (iid_z, "Independent generator", "#2374a5"),
                              (persistent_z, "Persistent generator", "#c54b26")]:
    axes[0].hist(values.ravel(), bins=bins, density=True, histtype="step", lw=1.8,
                 label=label, color=color)
    acf = [lag_correlation(values, lag).mean() for lag in lags]
    axes[1].plot(lags, acf, marker="o", color=color, label=label)
axes[0].set(xlabel="Standardized weather anomaly", ylabel="Probability density")
axes[1].set(xlabel="Lag (days)", ylabel="Anomaly correlation", xticks=lags, ylim=(-0.1, 1))
axes[1].axhline(0, color="#999999", lw=0.8)
axes[0].legend(fontsize=9)
plt.show()

<a id="response-histogram"></a>

**Discuss:** Can two generators have very similar histograms but produce different kinds of weather sequences? Use both panels as evidence.

**Answer:** [Your answer here.]

## 6. Generate an ensemble of hot spells

Call a day **hot** when its temperature exceeds the **95th percentile of training temperatures**. This is a classroom threshold, not an official heatwave definition.

For each generated five-year record, count hot days and find the longest consecutive run of hot days. The observed values below come from the five held-out years. Each point in the generated distributions represents one complete five-year record.

In [ ]:
hot_threshold = float(train.quantile(0.95))
print(f"Hot-day threshold from training data: {hot_threshold:.1f} °C")

def longest_hot_run(values):
    values = np.atleast_2d(values)
    run = np.zeros(values.shape[0], dtype=int)
    longest = np.zeros_like(run)
    for hot_today in (values > hot_threshold).T:
        run = np.where(hot_today, run + 1, 0)
        longest = np.maximum(longest, run)
    return longest

def statistics(temperature, z):
    temperature, z = np.atleast_2d(temperature), np.atleast_2d(z)
    return {
        "Hot days (five years)": (temperature > hot_threshold).sum(axis=1),
        "Longest hot spell (days)": longest_hot_run(temperature),
        "Lag-1 anomaly correlation": lag_correlation(z),
        "Standardized anomaly SD": z.std(axis=1),
    }

observed_stats = statistics(test.to_numpy(), obs_z)
iid_stats = statistics(iid_temperature, iid_z)
persistent_stats = statistics(persistent_temperature, persistent_z)

summary = pd.DataFrame({
    "Held-out observations": {k: v[0] for k, v in observed_stats.items()},
    "Independent generator: median": {k: np.median(v) for k, v in iid_stats.items()},
    "Persistent generator: median": {k: np.median(v) for k, v in persistent_stats.items()},
})
display(summary.round(3))

fig, axes = plt.subplots(1, 2, figsize=(11, 4), layout="constrained")
for ax, key in zip(axes, ["Hot days (five years)", "Longest hot spell (days)"]):
    low = min(iid_stats[key].min(), persistent_stats[key].min(), observed_stats[key][0])
    high = max(iid_stats[key].max(), persistent_stats[key].max(), observed_stats[key][0])
    bin_width = max(1, int(np.ceil((high-low+1)/20)))
    edges = np.arange(low-0.5, high+bin_width+0.5, bin_width)
    for values, color, label in [(iid_stats[key], "#2374a5", "Independent"),
                                  (persistent_stats[key], "#c54b26", "Persistent")]:
        ax.hist(values, bins=edges, alpha=0.45, color=color, label=label)
    ax.axvline(observed_stats[key][0], color="#111111", lw=2, label="Observed")
    ax.set(xlabel=key, ylabel="Number of generated records")
axes[0].legend(fontsize=9)
plt.show()

<a id="response-hot-spells"></a>

**Discuss:** How does persistence change the longest hot spells and the spread of hot-day counts? Which observed statistics does the persistent generator reproduce well, and which still need improvement?

**Answer:** [Your answer here.]

## Optional experiment: change persistence without changing the seasonal distribution

Change `PHI_EXPERIMENT`, rerun the cell, and inspect what changes. This is a sensitivity experiment: the observationally fitted value remains `phi_fit`.

Try `0.0`, the learned value, and `0.9`. The generator keeps the same normal marginal distribution while changing temporal dependence.

In [ ]:
PHI_EXPERIMENT = 0.9
experiment_temperature, experiment_z = generate_weather(phi=PHI_EXPERIMENT)
experiment_stats = statistics(experiment_temperature, experiment_z)
comparison = pd.DataFrame({
    "Independent: median": {k: np.median(v) for k, v in iid_stats.items()},
    "Learned persistence: median": {k: np.median(v) for k, v in persistent_stats.items()},
    f"Experiment phi={PHI_EXPERIMENT:g}: median": {k: np.median(v) for k, v in experiment_stats.items()},
})
display(comparison.round(3))

## Connection to generative AI

We fitted a probability model and sampled new weather from it. New random inputs give new realizations, even though the learned parameters stay fixed.

Our particular model specifies a Gaussian distribution and a one-day memory. Modern generative weather models can learn much more flexible distributions over entire atmospheric fields, often conditioned on recent weather states. For example, **GenCast** uses conditional diffusion to generate ensemble weather forecasts. The common idea is to learn a distribution and generate samples; the models differ in their representation, conditioning information, and sampling method.

The generator here represents a climate estimated from 1995–2009 observations. Its parameters do not describe an emissions-driven change in climate.

## Sources

- Data and starting point: [Lab 0c — Climate of New York City](https://dhruvbalwada.github.io/intro-climate-modeling-fall2026/labs/lab0c_nyc_climate.html). NOAA NCEI GHCN-Daily, Central Park station USW00094728; daily mean defined as (TMAX + TMIN)/2. The fixed course-file version and checksum are in `data/provenance.json`.
- Classical background: Richardson (1981), [Stochastic simulation of daily precipitation, temperature, and solar radiation](https://doi.org/10.1029/WR017i001p00182). This lab is a simplified temperature-only illustration.
- Modern example: Price et al. (2024), [Probabilistic weather forecasting with machine learning](https://doi.org/10.1038/s41586-024-08252-9); [GenCast explanation](https://deepmind.google/blog/gencast-predicts-weather-and-the-risks-of-extreme-conditions-with-sota-accuracy/).